## Set Operations:

- Set operations is used to `combine or compare the rows of two DataFrames`. 
- Common operations include `union(), unionByName(), intersect(), subtract(), and exceptAll()`.


Suppose an e-commerce company receives sales data from two sources:

- Chennai branch sales
- Madurai branch sales

You need to combine both branches' records into one DataFrame or identify records that appear in one dataset but not the other.


### Example 1: Create DataFrames

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SetOperations").getOrCreate()

chennai_df = spark.createDataFrame([
    (101, "Arun", 5000),
    (102, "Bala", 6000),
    (103, "Kumar", 7000)
], ["EmployeeID", "Name", "Salary"])

madurai_df = spark.createDataFrame([
    (102, "Bala", 6000),
    (103, "Kumar", 7000),
    (104, "Ravi", 8000)
], ["EmployeeID", "Name", "Salary"])


## A. union():

- **union()** is used to `combines the rows of two DataFrames`. 
- It `does not remove duplicate rows` automatically.

In [0]:
chennai_df.union(madurai_df).show()

## Combine daily sales files or records from multiple branches.

To remove `duplicate` rows:

In [0]:
chennai_df.union(madurai_df).distinct().show()

## B. unionByName():

**unionByName()** is used to `combines the rows of two DataFrames by matching column names`.

In [0]:
df1 = spark.createDataFrame([
    (101, "Arun")
], ["EmployeeID", "Name"])

df2 = spark.createDataFrame([
    ("Bala", 102)
], ["Name", "EmployeeID"])

df1.unionByName(df2).show()


## C. intersect():

**intersect()** is used to `returns the distinct rows and that rows are common to both DataFrames`.

In [0]:
chennai_df.intersect(madurai_df).show()


## D. subtract():

- **subtract()** is used to `returns the distinct rows from the first DataFrame` and `that rows are not appear in the second DataFrame`. 
- It behaves like a `set difference`.

In [0]:
chennai_df.subtract(madurai_df).show()


## E. exceptAll():

- **exceptAll()** is used to `returns rows from the first DataFrame` and `that rows are not matched by rows in the second DataFrame`.
- preserving `duplicate counts`.

In [0]:
chennai_df.exceptAll(madurai_df).show()

## Real-time scenario:


**An e-commerce company compares its previous customer snapshot with today's customer snapshot.**

- **old_customers_df:** 1,000,000 customer records.
- **new_customers_df:** 1,000,000 customer records.

- Customer IDs `500000–999999` exist in both datasets.


## Create large DataFrames in Databricks.

In [0]:
from pyspark.sql import functions as F

# Previous customer snapshot: 1 million records
old_customers_df = (
    spark.range(0, 1_000_000)
    .select(
        F.col("id").alias("CustomerID"),
        F.when(F.col("id") % 3 == 0, "Chennai")
         .when(F.col("id") % 3 == 1, "Madurai")
         .otherwise("Coimbatore")
         .alias("City")
    )
)

# Today's customer snapshot: 1 million records
new_customers_df = (
    spark.range(500_000, 1_500_000)
    .select(
        F.col("id").alias("CustomerID"),
        F.when(F.col("id") % 3 == 0, "Chennai")
         .when(F.col("id") % 3 == 1, "Madurai")
         .otherwise("Coimbatore")
         .alias("City")
    )
)

print("Old customers:", old_customers_df.count())
print("New customers:", new_customers_df.count())

old_customers_df.show(5)
new_customers_df.show(5)


## union() — Combine customer records

**Combine customer records received from two regional systems into one dataset.**

In [0]:
all_customers_df = old_customers_df.union(new_customers_df)

print("Total rows:", all_customers_df.count())
all_customers_df.show(5)


## intersect() — Find common customers:

**Identify customers present in both yesterday's and today's snapshots.**

In [0]:
common_customers_df = old_customers_df.intersect(new_customers_df)

print("Common rows:", common_customers_df.count())
common_customers_df.show(5)


## exceptAll() — Find records missing from today's data:

**Detect customer records present in yesterday's snapshot but missing from today's snapshot.**

In [0]:
missing_customers_df = old_customers_df.exceptAll(new_customers_df)

print("Missing rows:", missing_customers_df.count())
missing_customers_df.show(5)

## exceptAll() — Find distinct missing records:

**Generate a unique list of customers that existed previously but are no longer in today's snapshot.**

In [0]:
unique_missing_df = old_customers_df.exceptAll(
    new_customers_df
).distinct()

print("Distinct missing rows:", unique_missing_df.count())
unique_missing_df.show(5)


## intersectAll() — Compare matching records with duplicate counts:

**Reconcile transaction exports from a payment gateway and a bank. Repeated rows can represent repeated transaction records, so duplicate counts may matter.**

In [0]:
gateway_df = spark.createDataFrame(
    [
        (101, 500),
        (101, 500),
        (102, 700),
        (103, 900)
    ],
    ["TransactionID", "Amount"]
)

bank_df = spark.createDataFrame(
    [
        (101, 500),
        (102, 700),
        (102, 700),
        (104, 300)
    ],
    ["TransactionID", "Amount"]
)

common_transactions_df = gateway_df.intersectAll(bank_df)

common_transactions_df.show()


## unionByName() — Combine DataFrames with different column orders.

**One system exports CustomerID, City; another exports City, CustomerID. unionByName() matches columns by their names rather than their positions.**

In [0]:
system_a_df = spark.createDataFrame(
    [(101, "Chennai"), (102, "Madurai")],
    ["CustomerID", "City"]
)

system_b_df = spark.createDataFrame(
    [("Coimbatore", 103), ("Chennai", 104)],
    ["City", "CustomerID"]
)

combined_df = system_a_df.unionByName(system_b_df)

combined_df.show()


# Step 2: Create the Old Customer DataFrame

```python
old_customers_df = (
    spark.range(0, 1_000_000)
    .select(
        F.col("id").alias("CustomerID"),
        F.when(F.col("id") % 3 == 0, "Chennai")
         .when(F.col("id") % 3 == 1, "Madurai")
         .otherwise("Coimbatore")
         .alias("City")
    )
)
```

## 2.1 — `spark.range(0, 1_000_000)`

```python
spark.range(0, 1_000_000)
```

**Explanation:**

- Starting ID: `0`
- Ending ID: `999999`
- Total records: `1,000,000`
- The ending value `1_000_000` is excluded.

**Sample Output:**

| id |
|---:|
| 0 |
| 1 |
| 2 |
| 3 |
| 4 |
| ... |
| 999999 |

**Interview Definition:** `spark.range()` creates a DataFrame containing sequential numeric IDs within a specified range.

## 2.2 — `F.col()` and `alias()`

```python
F.col("id").alias("CustomerID")
```

**Explanation:**

- `F.col("id")` references the existing `id` column.
- `.alias("CustomerID")` renames the column to `CustomerID`.

**Before Renaming:**

| id |
|---:|
| 0 |
| 1 |
| 2 |

**After Renaming:**

| CustomerID |
|---:|
| 0 |
| 1 |
| 2 |

**Interview Definition:** `col()` references a DataFrame column, while `alias()` assigns a new name to a column or expression.

## 2.3 — `F.when()` and `.otherwise()`

```python
F.when(F.col("id") % 3 == 0, "Chennai")
 .when(F.col("id") % 3 == 1, "Madurai")
 .otherwise("Coimbatore")
 .alias("City")
```

**Explanation:**

The city is assigned based on the remainder when the customer ID is divided by 3.

- `%` is the modulo operator that returns the remainder.
- `when()` specifies a condition and its result.
- `otherwise()` specifies the fallback result when no preceding condition matches.
- `alias("City")` names the resulting column `City`.

**Example:**

| ID | ID % 3 | City |
|---:|---:|---|
| 0 | 0 | Chennai |
| 1 | 1 | Madurai |
| 2 | 2 | Coimbatore |
| 3 | 0 | Chennai |
| 4 | 1 | Madurai |
| 5 | 2 | Coimbatore |

The pattern repeats every three IDs.

**Interview Definition:** `when()` implements conditional logic in a DataFrame column, similar to SQL `CASE WHEN` or `IF...ELSE` logic.

## 2.4 — `select()`

```python
.select(
    F.col("id").alias("CustomerID"),
    ...
)
```

**Explanation:**

`select()` specifies the columns or expressions to include in the output DataFrame.

In this example:

- The original `id` column is renamed to `CustomerID`.
- A new `City` column is generated.

**Final Schema:**

```text
root
 |-- CustomerID: long
 |-- City: string
```

**Interview Definition:** `select()` retrieves specific columns or expressions from a DataFrame.

---

# Step 3: Create Today's Customer DataFrame

```python
new_customers_df = (
    spark.range(500_000, 1_500_000)
    .select(
        F.col("id").alias("CustomerID"),
        F.when(F.col("id") % 3 == 0, "Chennai")
         .when(F.col("id") % 3 == 1, "Madurai")
         .otherwise("Coimbatore")
         .alias("City")
    )
)
```

**Explanation:**

The logic is identical to the old DataFrame. Only the ID range changes.

```python
spark.range(500_000, 1_500_000)
```

- Starting ID: `500000`
- Ending ID: `1499999`
- Total records: `1,000,000`

**Sample Output:**

| CustomerID | City |
|---:|---|
| 500000 | Coimbatore |
| 500001 | Chennai |
| 500002 | Madurai |
| 500003 | Coimbatore |
| 500004 | Chennai |

The city values follow the same modulo rule explained in Step 2.3.

## Why Do the DataFrames Have Overlapping IDs?

The old DataFrame contains IDs from `0` to `999999`.

Today's DataFrame contains IDs from `500000` to `1499999`.

| Customer Group | Customer ID Range | Number of IDs |
|---|---|---:|
| Old-only customers | 0–499999 | 500,000 |
| Shared customers | 500000–999999 | 500,000 |
| New-only customers | 1000000–1499999 | 500,000 |